# Giai đoạn 2 — Mục 2.1, 2.3, 2.4 — Huấn luyện và đánh giá CNN raw và envelope với LOLO
**Đầu ra**:
- `outputs/tables/cnn_raw_lolo_results.csv`
- `outputs/tables/cnn_env_lolo_results.csv`
- Các model `.h5` và `.tflite` trong `outputs/models/`

## Phạm vi

**MLP là pipeline chính.** CNN1D raw/envelope chỉ được chạy như **thí nghiệm mở rộng** để so sánh tài nguyên (số tham số, kích thước `.tflite`, kết quả INT8, lịch sử huấn luyện). CNN không phải RQ chính.

Đối với CNN-envelope, chuẩn hóa dùng `StandardScaler` fit trên toàn bộ giá trị của **Train trong từng LOLO fold**, sau đó transform Val/Test bằng cùng scaler. Không dùng scaler toàn cục của toàn dataset.


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle
from sklearn.metrics import f1_score
from tensorflow.keras.callbacks import EarlyStopping

from common import models, quantization, training

In [13]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

windows_raw_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_raw.parquet")
windows_env_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/windows_cnn_env.parquet")

In [14]:
manifest_filtered = pd.read_csv("../giai_doan_1_tien_xu_ly/outputs/tables/manifest_filtered.csv")

# Hàm trích xuất dữ liệu cho CNN

In [15]:
def get_cnn_data(df, window_col='window', scaler=None, fit_scaler=False, standardize=False):
    X = np.stack(df[window_col].values).astype(np.float32)

    if standardize:
        # Fit scaler CHỈ trên Train của fold, sau đó tái sử dụng cho Val/Test.
        if fit_scaler:
            scaler = StandardScaler()
            scaler.fit(X.reshape(-1, 1))
        if scaler is None:
            raise ValueError("Cần scaler khi standardize=True.")
        X = scaler.transform(X.reshape(-1, 1)).reshape(X.shape).astype(np.float32)

    X = X[..., np.newaxis]
    y = df['label'].values
    return X, y, scaler


def audit_envelope_normalization(X_train, X_val, X_test):
    print("Envelope normalization audit")
    print(f"Train mean/std: {X_train.mean():.6f} / {X_train.std():.6f}")
    print(f"Val   mean/std: {X_val.mean():.6f} / {X_val.std():.6f}")
    print(f"Test  mean/std: {X_test.mean():.6f} / {X_test.std():.6f}")


# 1DCNN (raw/env) - LOLO

In [16]:
def run_cnn_lolo(df, window_col, build_model_func, window_size, model_name, standardize_envelope=False):
    results = []
    history_rows = []
    param_rows = []

    for fold_info, train_df, val_df, test_df in training.iterate_lolo_splits(df, load_col='load_hp'):
        print(f"\n--- {model_name} - Fold: {fold_info['fold_name']} ---")

        X_train, y_train, scaler = get_cnn_data(
            train_df, window_col,
            scaler=None, fit_scaler=standardize_envelope,
            standardize=standardize_envelope
        )
        X_val, y_val, _ = get_cnn_data(
            val_df, window_col,
            scaler=scaler, fit_scaler=False,
            standardize=standardize_envelope
        )
        X_test, y_test, _ = get_cnn_data(
            test_df, window_col,
            scaler=scaler, fit_scaler=False,
            standardize=standardize_envelope
        )

        if standardize_envelope:
            audit_envelope_normalization(X_train, X_val, X_test)

        le = LabelEncoder()
        y_train_enc = le.fit_transform(y_train)
        y_val_enc = le.transform(y_val)
        y_test_enc = le.transform(y_test)

        model = build_model_func(window_size=window_size)
        model = models.compile_classifier(model)
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        history = model.fit(
            X_train, y_train_enc,
            validation_data=(X_val, y_val_enc),
            epochs=100, batch_size=64,
            callbacks=[early_stop], verbose=0
        )

        for epoch_idx in range(len(history.history['loss'])):
            history_rows.append({
                'model': model_name,
                'fold': fold_info['fold_name'],
                'epoch': epoch_idx + 1,
                'loss': history.history['loss'][epoch_idx],
                'val_loss': history.history['val_loss'][epoch_idx],
                'accuracy': history.history['accuracy'][epoch_idx],
                'val_accuracy': history.history['val_accuracy'][epoch_idx],
            })

        loss, acc = model.evaluate(X_test, y_test_enc, verbose=0)
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        f1 = f1_score(y_test_enc, y_pred, average='macro')

        tflite_bytes = quantization.quantize_model_int8(model, X_train)
        int8_result = quantization.evaluate_tflite_model(tflite_bytes, X_test, y_test_enc)

        param_count = int(model.count_params())
        tflite_size_bytes = int(len(tflite_bytes))
        tflite_size_kb = tflite_size_bytes / 1024.0

        results.append({
            'fold': fold_info['fold_name'],
            'test_load': fold_info['test_load'],
            'float_accuracy': acc,
            'float_f1': f1,
            'int8_accuracy': int8_result['accuracy'],
            'epochs': len(history.history['loss']),
            'best_val_loss': float(np.min(history.history['val_loss'])),
            'best_val_accuracy': float(np.max(history.history['val_accuracy'])),
            'num_parameters': param_count,
            'tflite_size_bytes': tflite_size_bytes,
            'tflite_size_kb': tflite_size_kb
        })

        param_rows.append({
            'model': model_name,
            'fold': fold_info['fold_name'],
            'num_parameters': param_count,
            'tflite_size_bytes': tflite_size_bytes,
            'tflite_size_kb': tflite_size_kb
        })

        model.save(MODELS_DIR / f"{model_name}_{fold_info['fold_name']}.h5")
        quantization.model_bytes_to_file(tflite_bytes, MODELS_DIR / f"{model_name}_{fold_info['fold_name']}.tflite")

        if standardize_envelope:
            with open(MODELS_DIR / f"scaler_{model_name}_{fold_info['fold_name']}.pkl", 'wb') as f:
                pickle.dump(scaler, f)

    pd.DataFrame(history_rows).to_csv(TABLES_DIR / f"{model_name}_training_history_per_epoch.csv", index=False)
    pd.DataFrame(param_rows).to_csv(TABLES_DIR / f"{model_name}_model_size.csv", index=False)
    return pd.DataFrame(results)


In [17]:
print("Các cột của manifest_filtered:", manifest_filtered.columns)
print("Index của manifest_filtered:", manifest_filtered.index.name)
display(manifest_filtered.head())

Các cột của manifest_filtered: Index(['file_path', 'load_hp', 'label', 'fault_diameter_mils', 'or_position',
       'source_category', 'sensor_location', 'declared_sample_rate_khz',
       'n_samples_DE', 'n_samples_FE', 'n_samples_BA', 'rpm_from_file',
       'read_error', 'warnings', 'has_warning'],
      dtype='str')
Index của manifest_filtered: None


,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,NaN,NaN,False
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,NaN,NaN,False
2,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,2,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1748.0,NaN,NaN,False
3,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,3,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1722.0,NaN,NaN,False
4,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,14.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121846,121846.0,121846.0,1796.0,NaN,NaN,False


In [18]:
# 1. Khôi phục lại cột file_id cho manifest_filtered (Khớp định dạng với file 04 Giai đoạn 1)
manifest_filtered['file_id'] = manifest_filtered.apply(
    lambda row: f"{row['label']}_{row['load_hp']}_{row.get('fault_diameter_mils', '')}_{row['file_path']}", axis=1
)

# 2. Lấy metadata và thực hiện Merge
metadata = manifest_filtered[['file_id', 'load_hp']]

windows_raw_df = windows_raw_df.merge(metadata, on='file_id', how='left')
windows_env_df = windows_env_df.merge(metadata, on='file_id', how='left')

# Loại bỏ các dòng NaN (nếu có do merge không khớp)
windows_raw_df = windows_raw_df.dropna(subset=['load_hp']).reset_index(drop=True)
windows_env_df = windows_env_df.dropna(subset=['load_hp']).reset_index(drop=True)

# 3. Ép kiểu load_hp về int để hàm training.iterate_lolo_splits hoạt động đúng
windows_raw_df['load_hp'] = windows_raw_df['load_hp'].astype(int)
windows_env_df['load_hp'] = windows_env_df['load_hp'].astype(int)

print("Các cột của windows_raw_df sau khi merge:", windows_raw_df.columns)

Các cột của windows_raw_df sau khi merge: Index(['file_id', 'label', 'start_idx', 'window', 'load_hp'], dtype='str')


In [19]:
# CNN chỉ là thí nghiệm mở rộng để so sánh tài nguyên với MLP.
cnn_raw_results = run_cnn_lolo(
    windows_raw_df, 'window',
    models.build_cnn1d_raw, 2048, 'cnn_raw',
    standardize_envelope=False
)

cnn_env_results = run_cnn_lolo(
    windows_env_df, 'window',
    models.build_cnn1d_env, 1024, 'cnn_env',
    standardize_envelope=True
)

cnn_raw_results.to_csv(TABLES_DIR / "cnn_raw_lolo_results.csv", index=False)
cnn_env_results.to_csv(TABLES_DIR / "cnn_env_lolo_results.csv", index=False)

print("Đã lưu history và model size cho CNN raw/envelope.")



--- cnn_raw - Fold: test_load_0 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpdu9clhku\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpdu9clhku\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpdu9clhku'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2048, 1), dtype=tf.float32, name='raw_signal')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1994015891792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995339251984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995339256592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995339261392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1993004155536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1993004157264: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



--- cnn_raw - Fold: test_load_1 ---
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmp1vj32x5t\assets


...


--- cnn_env - Fold: test_load_3 ---
Envelope normalization audit
Train mean/std: -0.000000 / 1.000000
Val   mean/std: -0.131579 / 0.149730
Test  mean/std: -0.031570 / 0.970097
INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpt7r5cj7x\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpt7r5cj7x\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpt7r5cj7x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024, 1), dtype=tf.float32, name='envelope')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1995925948176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925950672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925950288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925947408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925947984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925948560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925951056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925950864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925947792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925949328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1995925949136

f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Đã lưu history và model size cho CNN raw/envelope.


In [20]:
cnn_raw_results.head()

,fold,test_load,float_accuracy,float_f1,int8_accuracy,epochs,best_val_loss,best_val_accuracy,num_parameters,tflite_size_bytes,tflite_size_kb
0,test_load_0,0,0.868726,0.878172,0.776834,15,0.193164,0.889831,2652,8888,8.679688
1,test_load_1,1,0.989536,0.988630,0.842381,14,0.325802,0.817326,2652,8888,8.679688
2,test_load_2,2,0.998037,0.997869,0.846204,18,0.167057,0.938795,2652,8888,8.679688
3,test_load_3,3,0.861799,0.850503,0.741851,17,0.084004,0.998786,2652,8888,8.679688


In [21]:
cnn_env_results.head()

,fold,test_load,float_accuracy,float_f1,int8_accuracy,epochs,best_val_loss,best_val_accuracy,num_parameters,tflite_size_bytes,tflite_size_kb
0,test_load_0,0,0.957774,0.956225,0.763148,58,0.152214,0.980760,1444,8536,8.335938
1,test_load_1,1,0.888744,0.880718,0.652245,32,0.273346,0.967136,1444,8536,8.335938
2,test_load_2,2,0.897461,0.889438,0.532552,26,0.324316,0.947418,1444,8536,8.335938
3,test_load_3,3,0.917884,0.912784,0.567348,21,0.372542,0.908213,1444,8536,8.335938


In [22]:
# Tổng hợp tài nguyên MLP vs CNN phục vụ Bảng 2b/Bảng bổ sung
mlp_size_path = Path('../giai_doan_2_xay_dung_mo_hinh/outputs/tables/mlp_model_size.csv')
local_mlp_size = TABLES_DIR / 'mlp_model_size.csv'

size_frames = []
if local_mlp_size.exists():
    size_frames.append(pd.read_csv(local_mlp_size))
else:
    print(f'Chưa tìm thấy {local_mlp_size}; hãy chạy 02_mlp_lolo_revised.ipynb trước.')

for name in ['cnn_raw', 'cnn_env']:
    p = TABLES_DIR / f'{name}_model_size.csv'
    if p.exists():
        size_frames.append(pd.read_csv(p))

if size_frames:
    model_size_comparison = pd.concat(size_frames, ignore_index=True)
    model_size_comparison.to_csv(TABLES_DIR / 'model_size_comparison_mlp_cnn.csv', index=False)
    display(model_size_comparison)


,model,fold,num_parameters,tflite_size_bytes,tflite_size_kb
0,MLP,test_load_0,1652,5672,5.539062
1,MLP,test_load_1,1652,5672,5.539062
2,MLP,test_load_2,1652,5672,5.539062
3,MLP,test_load_3,1652,5672,5.539062
4,cnn_raw,test_load_0,2652,8888,8.679688
5,cnn_raw,test_load_1,2652,8888,8.679688
6,cnn_raw,test_load_2,2652,8888,8.679688
7,cnn_raw,test_load_3,2652,8888,8.679688
8,cnn_env,test_load_0,1444,8536,8.335938
9,cnn_env,test_load_1,1444,8536,8.335938
